# Maternal Stage EF — Pre vs Post (PD1/PD3/PD4) (1Hz frequency)

Paper-active dCSFA-NMF model for separating Pre-pup home windows from P1/P3/P4-home windows (1Hz frequency resolution).

* Final model: `Maternal_model_TrainC_Pre_P134_1Hz_ver2.pt`
* Hyperparameters from LOO validation (sup_weight=0.03, n_epochs=400, batch_size=512, h=128, seed=2025)
* Stage backproject artifact: `StageEF_1Hz.xlsx`

In [ ]:
# Allow imports from ../src
import sys, os, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if os.path.join(_repo_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_repo_root, "src"))

from electome.data_utils import create_period_dataset, create_split_dataset, clean_mouse_id
from electome.training import run_loo_cv, train_final_model
from electome.analysis import process_W_nmf_dual_filter
from electome.viz import create_bar_heatmap_selective
from electome.workflow import validate_on_ELS, run_circos_prep, run_stage_backproject
from electome.sara_requests import sara_pup_retrieval


## Section 1. Data loading and processing

Filter to Pre home / P1 / P3 / P4 home periods. Binary label: `Pre home → 0`, `P1/P3/P4 home → 1`. The C/E split is built via `create_period_dataset` and the `mouse_id` filtering keys are the underscored raw pkl form (e.g. `'C7_ELS11'`).

In [ ]:
TRAINING_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features_Trim_All.pkl"

# Raw mouse_ids with underscores (matches the pkl format)
C_MOUSE_IDS_RAW = ["C7_ELS11", "C2_ELS18", "C5_ELS20", "C7_ELS22",
                    "C1_ELS32", "C5_ELS40", "C6_ELS42", "C7_ELS45"]
E_MOUSE_IDS_RAW = ["E1_ELS33", "E2_ELS3", "E3_ELS37", "E4_ELS39",
                    "E5_ELS41", "E6_ELS44"]

# Cleaned ids for the stage-backproject section
C_MOUSE_IDS = [m.replace("_", "") for m in C_MOUSE_IDS_RAW]
E_MOUSE_IDS = [m.replace("_", "") for m in E_MOUSE_IDS_RAW]

PERIODS_TO_KEEP = ["Pre home", "P1", "P3", "P4 home"]
POSITIVE_PERIODS = ["P1", "P3", "P4 home"]   # mapped to y=1

with open(TRAINING_DATA_FILE, "rb") as f:
    train_dict = pickle.load(f)

base_mask = np.isin(train_dict["period"], PERIODS_TO_KEEP)
base_data = {k: v[base_mask] for k, v in train_dict.items()
             if isinstance(v, np.ndarray) and len(v) == len(train_dict["period"])}

# Binary y from period (Pre home -> 0, P1/P3/P4 home -> 1)
base_y = np.isin(base_data["period"], POSITIVE_PERIODS).astype(np.int64)

# train = all kept periods; held-out tests are computed by period subset on the SAME pkl
train_c     = create_period_dataset(base_data, base_y, C_MOUSE_IDS_RAW, PERIODS_TO_KEEP,   "Train C",      verbose=False)
test_e_all  = create_period_dataset(base_data, base_y, E_MOUSE_IDS_RAW, PERIODS_TO_KEEP,   "Test E (all)", verbose=False)
test_e_post = create_period_dataset(base_data, base_y, E_MOUSE_IDS_RAW, POSITIVE_PERIODS,  "Test E (post)", verbose=False)
test_e_pre  = create_period_dataset(base_data, base_y, E_MOUSE_IDS_RAW, ["Pre home"],      "Test E (Pre)",  verbose=False)

print(f"Train C       : X={train_c['X'].shape}, mice={list(train_c['mouse_list'])}")
print(f"Test E (all)  : X={test_e_all['X'].shape}, mice={list(test_e_all['mouse_list'])}")
print(f"Test E (post) : X={test_e_post['X'].shape}")
print(f"Test E (Pre)  : X={test_e_pre['X'].shape}")


## Section 2. LOO training

In [ ]:
SEED = 2025
N_EPOCHS = 400
N_PRE_EPOCHS = 100
NMF_MAX_ITER = 500
BATCH_SIZE = 512
LR = 0.001

MODEL_PARAMS = {
    "n_components": 10,
    "n_sup_networks": 1,
    "optim_name": "SGD",
    "recon_loss": "MSE",
    "sup_recon_weight": 0.0,
    "sup_weight": 0.03,
    "phi_weight": 0,
    "n_intercepts": 1,
    "useDeepEnc": True,
    "h": 128,
    "sup_recon_type": "Residual",
    "feature_groups": None,
    "group_weights": None,
    "fixed_corr": "Positive",
    "momentum": 0.9,
    "sup_smoothness_weight": 1,
}

loo = run_loo_cv(
    train_c["X"], train_c["y"], train_c["y_intercept"],
    model_params=MODEL_PARAMS,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    n_pre_epochs=N_PRE_EPOCHS, nmf_max_iter=NMF_MAX_ITER,
    seed=SEED, n_jobs=4,
)
print(loo.summary())
print()
print(loo.per_mouse_table())


## Section 3. Full training (paper model)

In [ ]:
MODEL_SAVE_FILE = "Maternal_model_TrainC_Pre_P134_1Hz_ver2.pt"
MODEL_STATE_DICT = "Maternal_sd_TrainC_Pre_P134_1Hz_ver2.pt"

model = train_final_model(
    train_c["X"], train_c["y"], train_c["y_sampling"],
    model_params=MODEL_PARAMS,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    n_pre_epochs=N_PRE_EPOCHS, nmf_max_iter=NMF_MAX_ITER,
    seed=SEED,
    save_to=MODEL_SAVE_FILE,
    state_dict_to=MODEL_STATE_DICT,
)
train_aucs = [auc[0] for auc in model.train_auc_hist]
print(f"Paper model saved : {MODEL_SAVE_FILE}")
print(f"  Final train AUC : {train_aucs[-1]:.4f}")


## Section 4. Circos plot

In [ ]:
df_circos = run_circos_prep(
    model, train_dict,
    output_csv="StageEF_1Hz_circos_input.csv",
    k=0, threshold_ratio=0.8,
)


## Section 5. Elements selection

In [ ]:
abs_cut, rel_cut, both_cut, abs_full, rel_full = process_W_nmf_dual_filter(
    model.get_W_nmf(), train_dict,
    abs_cum_ratio=0.9, rel_val=0.5,
    verbose=False,
)
fig = create_bar_heatmap_selective(abs_full, abs_cut, rel_full, rel_cut, both_cut)
fig.savefig("StageEF_1Hz_bar_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Bar heatmap saved : StageEF_1Hz_bar_heatmap.png  "
      f"({(~both_cut.isna()).sum().sum()} optimal features)")


## Section 6. Validation on ELS group

In [ ]:
els = validate_on_ELS(model, {
    "E mice (all PD periods)": test_e_all,
    "E mice (post = P1/P3/P4)": test_e_post,
    "E mice (Pre home only)":  test_e_pre,
})
print(els.summary())


## Section 7. Stage Backprojection Scores

In [ ]:
BACKPROJECT_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features_Trim_All.pkl"

backproject = run_stage_backproject(
    model,
    backproject_data_file=BACKPROJECT_DATA_FILE,
    c_mouse_ids=C_MOUSE_IDS,
    e_mouse_ids=E_MOUSE_IDS,
    output_xlsx="StageEF_1Hz.xlsx",
)
print(backproject.summary())


## Section 8. Additional backprojections (Sara's request)

In [ ]:
PUP_RETRIEVAL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P4_pup_retrieval_detail.pkl"

sara_pup_retrieval(
    model,
    pup_retrieval_data_file=PUP_RETRIEVAL_DATA_FILE,
    c_mouse_ids=["C6ELS9"] + C_MOUSE_IDS,
    e_mouse_ids=E_MOUSE_IDS,
    output_xlsx="StageEF_1Hz_pups.xlsx",
)
